# UR5 + Paddle Ping-Pong Juggling — MuJoCo environment

Colab setup, a rendered drop test, and the batched-rollout API an MPC planner uses.
Rendering is faster on a GPU runtime (`Runtime > Change runtime type > T4 GPU`), but the CPU runtime also works.

In [ ]:
# 1) Get the code. For a private repo, use https://<TOKEN>@github.com/... (keep the token in Colab Secrets).
REPO = "https://github.com/dnltjrqja-dev/cs285.git"
BRANCH = "claude/mpc-ping-pong-juggling-k3rhv6"
!git clone -q -b {BRANCH} {REPO} cs285 || (cd cs285 && git pull -q)
%cd cs285
!pip install -q -e ".[viz,test]"

In [ ]:
# 2) Headless rendering backend: EGL on GPU runtimes, OSMesa on CPU runtimes.
import os, subprocess
if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0:
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq install -y libosmesa6 > /dev/null
    os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = os.environ["MUJOCO_GL"]
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

In [ ]:
import time
import numpy as np
import mediapy as media
from ppjuggle import UR5PaddleJuggleEnv, EnvConfig, HOME_QPOS

env = UR5PaddleJuggleEnv(seed=0)
obs, info = env.reset()
print("obs dim:", obs.shape, "| control dt:", env.dt, "s")
print("paddle center:", env.paddle_pos.round(3), "normal:", env.paddle_normal.round(3))
media.show_image(env.render())

## Drop test: the paddle stays still and the ball bounces on it
Restitution is about 0.85, and air drag (Cd ≈ 0.47) plus Magnus lift are on.
The ball stops bouncing because nothing is controlling the paddle yet.

In [ ]:
env.reset(seed=0)
frames = []
for t in range(300):
    _, _, terminated, _, info = env.step(np.zeros(6))
    if t % 2 == 0:
        frames.append(env.render())
    if terminated:
        break
print("hits:", info["num_hits"], "| ball dropped:", terminated, "| sim time:", round(info["time"], 2), "s")
media.show_video(frames, fps=50)

## Batched rollouts for MPC
`env.rollout(U)` simulates `K` candidate sequences of joint targets `U` with shape `(K, H, 6)`. It runs them in parallel threads, starting from the current state.
It returns the full-physics states and the sensors at the end of every control step: paddle pos/normal/vel and ball pos/vel.
`env.data` is not changed, so an MPPI or CEM planner can call it every control step.

In [ ]:
K, H = 256, 40   # 256 samples, 0.4 s horizon at 100 Hz
U = HOME_QPOS + 0.03 * np.random.randn(K, H, 6)
t0 = time.time()
states, sensors = env.rollout(U)
print(f"{K} x {H} control steps ({K*H*env.cfg.frame_skip:,} physics steps) in {time.time()-t0:.3f} s")
sl = env.sensor_slices()
ball_z = sensors[:, :, sl["ball_pos"]][..., 2]
print("ball z over horizon, sample 0:", ball_z[0, ::8].round(3))

## Where the MPC controller goes
```python
for t in range(T):
    U = planner.plan(env)            # uses env.rollout(...) internally
    obs, r, term, trunc, info = env.step(U[0])   # with EnvConfig(action_mode="absolute")
```

In [ ]:
!python -m pytest -q tests